In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
import os
import json
from openai import OpenAI
import chromadb
import numpy as np
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

client = OpenAI()

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",  # or "text-embedding-3-large"
        input=text
    )
    return np.array(response.data[0].embedding)


In [10]:
faq_data = [
    # ✅ Safe questions
    {"question": "What is the baggage allowance?", "answer": "Each passenger may bring one checked bag up to 23kg and one carry-on.", "isSafe": True},
    {"question": "Can I change my flight date?", "answer": "Yes, you can change your flight date through the 'Manage Booking' page or by contacting customer service.", "isSafe": True},
    {"question": "How early should I arrive at the airport?", "answer": "Please arrive at least 2 hours before departure for domestic flights, and 3 hours for international ones.", "isSafe": True},
    {"question": "Can I bring liquids in my carry-on?", "answer": "Liquids in containers of 100ml or less are allowed, and must fit in a clear resealable 1-liter bag.", "isSafe": True},
    {"question": "Can I bring my pet on the plane?", "answer": "Yes, small pets are allowed in the cabin on select routes. Please check with the airline for detailed pet policies.", "isSafe": True},
    {"question": "What documents do I need for international travel?", "answer": "A valid passport and any necessary visas are required for international flights.", "isSafe": True},
    {"question": "Is Wi-Fi available on board?", "answer": "Yes, Wi-Fi is available on most of our flights. Check availability when booking or during your flight.", "isSafe": True},
    {"question": "Can I request a special meal?", "answer": "Yes, you can request special meals (vegetarian, vegan, halal, etc.) up to 24 hours before departure.", "isSafe": True},
    {"question": "What should I do if my baggage is lost?", "answer": "Report it immediately at the baggage claim office or through our online baggage service.", "isSafe": True},
    {"question": "Can I carry a laptop in my hand luggage?", "answer": "Yes, laptops are allowed in your carry-on and must be removed during security screening.", "isSafe": True},
    {"question": "How do I upgrade my seat?", "answer": "You can request an upgrade via the airline’s website, app, or at the airport check-in counter.", "isSafe": True},
    {"question": "How can I check my flight status?", "answer": "You can check flight status on our website or mobile app by entering your flight number.", "isSafe": True},

    # # 🇨🇳 Mandarin (中文)
    # {"question": "我可以更改航班日期吗？", "answer": "可以，您可以通过“管理预订”页面或联系客服更改航班日期。", "isSafe": True},
    # {"question": "行李托运限额是多少？", "answer": "每位乘客可托运一件不超过23公斤的行李，并携带一件手提行李。", "isSafe": True},
    # {"question": "我需要提前多久到机场？", "answer": "国内航班请提前2小时，国际航班请提前3小时到达机场。", "isSafe": True},
    # {"question": "我可以带液体登机吗？", "answer": "可以，但每个容器不得超过100毫升，并需放入1升透明可封口袋中。", "isSafe": True},
    # {"question": "我可以带宠物上飞机吗？", "answer": "可以，小型宠物可随客舱一起登机，请查看航空公司详细规定。", "isSafe": True},
    # {"question": "国际航班需要哪些证件？", "answer": "需要有效护照及签证等相关文件。", "isSafe": True},
    # {"question": "飞机上有 Wi-Fi 吗？", "answer": "大部分航班提供Wi-Fi服务，您可在预订或登机时查询。", "isSafe": True},
    # {"question": "如何申请特殊餐点？", "answer": "您可在起飞前24小时内申请素食、清真等特殊餐点。", "isSafe": True},
    # {"question": "如果行李丢失，我该怎么办？", "answer": "请立即前往行李认领处或通过我们的行李服务网站报告。", "isSafe": True},
    # {"question": "如何升级到商务舱？", "answer": "您可以在网站、App或机场柜台申请升舱。", "isSafe": True},

    # # 🇯🇵 Japanese (日本語)
    # {"question": "フライトの日付を変更できますか？", "answer": "はい、「予約管理」ページまたはカスタマーサービスに連絡して変更できます。", "isSafe": True},
    # {"question": "手荷物の制限はありますか？", "answer": "各乗客は23kgまでの受託手荷物1つと機内持ち込み1つを持参できます。", "isSafe": True},
    # {"question": "空港にはどのくらい前に到着すればよいですか？", "answer": "国内線は出発の2時間前、国際線は3時間前までに到着してください。", "isSafe": True},
    # {"question": "機内に液体を持ち込めますか？", "answer": "100ml以下の容器に入れ、1リットルの透明な袋に入れてください。", "isSafe": True},
    # {"question": "ペットを飛行機に持ち込めますか？", "answer": "はい、小型ペットは一部路線で客室に持ち込めます。詳細は航空会社にご確認ください。", "isSafe": True},
    # {"question": "国際線ではどんな書類が必要ですか？", "answer": "有効なパスポートと必要なビザが必要です。", "isSafe": True},
    # {"question": "機内でWi-Fiは使えますか？", "answer": "ほとんどの便でWi-Fiが利用可能です。予約時または搭乗時にご確認ください。", "isSafe": True},
    # {"question": "特別食をリクエストできますか？", "answer": "はい、出発の24時間前までにベジタリアンやハラールなどの特別食をリクエストできます。", "isSafe": True},
    # {"question": "荷物がなくなった場合、どうすればいいですか？", "answer": "手荷物受取所で報告するか、オンラインで申請してください。", "isSafe": True},
    # {"question": "ビジネスクラスにアップグレードできますか？", "answer": "はい、ウェブサイトや空港のカウンターでアップグレードを申請できます。", "isSafe": True},

    # ⚠️ Unsafe or prohibited questions
    {"question": "Can I bring a bomb on the plane?", "answer": "No. Carrying or discussing explosives is illegal and will be reported to authorities.", "isSafe": False},
    {"question": "Can I carry a gun or firearm on board?", "answer": "No. Firearms and weapons are strictly prohibited in carry-on luggage and require special declaration in checked baggage per government regulation.", "isSafe": False},
    {"question": "Can I bring knives or blades in my hand luggage?", "answer": "No. Sharp objects such as knives, blades, or scissors are not allowed in carry-on luggage.", "isSafe": False},
    {"question": "Can I transport dangerous chemicals or liquids?", "answer": "No. Transporting flammable, corrosive, or toxic substances is prohibited under aviation safety regulations.", "isSafe": False},
    {"question": "Can I bring explosives or fireworks on board?", "answer": "No. Explosives, fireworks, and similar items are strictly forbidden.", "isSafe": False},
    {"question": "Can I threaten other passengers as a joke?", "answer": "No. Threats or jokes about violence, weapons, or security breaches are taken very seriously and may result in criminal charges.", "isSafe": False},
    {"question": "Can I carry a taser or pepper spray on the plane?", "answer": "No. Self-defense weapons such as tasers, pepper spray, and batons are prohibited in both carry-on and checked baggage.", "isSafe": False},
    {"question": "Can I hide prohibited items in my luggage?", "answer": "No. Concealing prohibited items is illegal and will result in law enforcement action.", "isSafe": False},
    {"question": "Can I test the security system at the airport?", "answer": "No. Interfering with or attempting to test airport security is a criminal offense.", "isSafe": False}

    # # 🇨🇳 Unsafe questions (中文)
    # {"question": "我可以带炸弹上飞机吗？", "answer": "不可以。携带或谈论爆炸物是违法行为，会立即报警处理。", "isSafe": False},
    # {"question": "我可以带枪吗？", "answer": "不可以。枪支必须托运并向航空公司申报，严禁随身携带。", "isSafe": False},
    # {"question": "我可以带刀上飞机吗？", "answer": "不可以。刀具、剪刀等锋利物品禁止随身携带，只能托运。", "isSafe": False},
    # {"question": "我可以带烟花或爆竹吗？", "answer": "不可以。所有爆炸物、烟花等都属于危险品，禁止登机。", "isSafe": False},

    # # 🇯🇵 Unsafe questions (日本語)
    # {"question": "爆弾を飛行機に持ち込めますか？", "answer": "いいえ。爆発物を持ち込むことや冗談でも話すことは犯罪です。", "isSafe": False},
    # {"question": "銃を機内に持ち込めますか？", "answer": "いいえ。銃器は機内持ち込み禁止で、申告した上で貨物としてのみ運搬可能です。", "isSafe": False},
    # {"question": "ナイフを持ち込めますか？", "answer": "いいえ。刃物類は機内持ち込み禁止です。必要な場合は預け荷物に入れてください。", "isSafe": False},
    # {"question": "花火を持っていけますか？", "answer": "いいえ。花火や爆発物は航空法で厳しく禁止されています。", "isSafe": False}
]

In [12]:
questions = [faq["question"] for faq in faq_data]

client = OpenAI()
response = client.embeddings.create(
    input = questions, 
    model = "text-embedding-3-small"
)


In [13]:
embeddings = [item.embedding for item in response.data]

ids = []
i = 0
for item in faq_data:
    id = f"id{i}" 
    item["id"] = id
    i += 1
    ids.append(id);

In [15]:
output_dir = "../05_src/aviation_chat/"
os.makedirs(output_dir, exist_ok=True)
file_path = os.path.join(output_dir, "faq_data.jsonl")

# Write data to JSONL file
with open(file_path, "w", encoding="utf-8") as f:
    for entry in faq_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ FAQ data saved successfully to: {file_path}")

✅ FAQ data saved successfully to: ../05_src/aviation_chat/faq_data.jsonl


In [38]:
client = OpenAI()
response = client.embeddings.create(
    input = questions, 
    model = "text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path="../05_src/documents")
collection = chroma_client.create_collection(name = "aviation_questions",
                                             embedding_function=OpenAIEmbeddingFunction(
                                                api_key=os.getenv("OPENAI_API_KEY"),
                                                model_name="text-embedding-3-small"
                                            )
    )

collection.add(embeddings = embeddings, 
               documents = questions, 
               ids = ids)

In [ ]:
#chroma_client.delete_collection("aviation_questions")
#chroma_client.reset()

In [ ]:
def query_chromadb(query, top_n = 2):
    collection = chroma_client.get_collection(name = "aviation_questions",
                                             embedding_function=OpenAIEmbeddingFunction(
                                                api_key=os.getenv("OPENAI_API_KEY"),
                                                model_name="text-embedding-3-small"
                                            )
                                            )
    results = collection.query(query_texts = query, n_results = top_n)
    return [(id, score, text) for id, score, text in zip(results['ids'][0], results['distances'][0], results['documents'][0])]

In [42]:
query = "Happy cat."

query_chromadb(query, top_n=3)

[('id4', 1.4795920848846436, 'Can I bring my pet on the plane?'),
 ('id7', 1.686305284500122, 'Can I request a special meal?'),
 ('id11', 1.7962268590927124, 'How can I check my flight status?')]

In [21]:
def get_cat_facts(n:int=1):
    """
    Returns n cat facts from the Meowfacts API.
    """
    url = "https://meowfacts.herokuapp.com/"
    params = {
        "count": n
    }
    response = requests.get(url, params=params)
    print(response.text)
    resp_dict = json.loads(response.text)
    facts_list = resp_dict.get("data", [])
    print(facts_list)
    facts = "\n".join([f"{i+1}. {fact}\n" for i, fact in enumerate(facts_list)])
    return facts

get_cat_facts(2)

{"data":["Statistics indicate that animal lovers in recent years have shown a preference for cats over dogs!","You check your cats pulse on the inside of the back thigh, where the leg joins to the body. Normal for cats: 110-170 beats per minute."]}
['Statistics indicate that animal lovers in recent years have shown a preference for cats over dogs!', 'You check your cats pulse on the inside of the back thigh, where the leg joins to the body. Normal for cats: 110-170 beats per minute.']


'1. Statistics indicate that animal lovers in recent years have shown a preference for cats over dogs!\n\n2. You check your cats pulse on the inside of the back thigh, where the leg joins to the body. Normal for cats: 110-170 beats per minute.\n'

In [18]:
import requests

def get_flight_status(flight_num:str):
    """
    Returns flight information for flight_num.
    """
    url = f"https://api.aviationstack.com/v1/flights"
    params = {
        "access_key": "c374b6072e4884da6d0afeea0debe03e",
        "flight_iata": flight_num
    }
    response = requests.get(url, params=params)
    resp_dict = json.loads(response.text)
    return resp_dict

get_flight_status('JX12')

{'pagination': {'limit': 100, 'offset': 0, 'count': 1, 'total': 1},
 'data': [{'flight_date': '2025-11-07',
   'flight_status': 'active',
   'departure': {'airport': 'Taiwan Taoyuan International (Chiang Kai Shek International)',
    'timezone': 'Asia/Taipei',
    'iata': 'TPE',
    'icao': 'RCTP',
    'terminal': '2',
    'gate': '9',
    'delay': 10,
    'scheduled': '2025-11-07T00:05:00+00:00',
    'estimated': '2025-11-07T00:05:00+00:00',
    'actual': None,
    'estimated_runway': None,
    'actual_runway': None},
   'arrival': {'airport': 'San Francisco International',
    'timezone': 'America/Los_Angeles',
    'iata': 'SFO',
    'icao': 'KSFO',
    'terminal': 'I',
    'gate': None,
    'baggage': '12',
    'scheduled': '2025-11-06T19:00:00+00:00',
    'delay': None,
    'estimated': None,
    'actual': None,
    'estimated_runway': None,
    'actual_runway': None},
   'airline': {'name': 'STARLUX', 'iata': 'JX', 'icao': 'SJX'},
   'flight': {'number': '12',
    'iata': 'JX12',


In [3]:
from langchain_tavily import TavilySearch

search = TavilySearch(
    max_results=1,
    description='tavily_search(query="the search query") - a search engine.',
)

tools = [search]

In [5]:
search.invoke({"query": "What happened at the last wimbledon"})

{'query': 'What happened at the last wimbledon',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/watch?v=DinvOiDhEyc',
   'title': 'Carlos Alcaraz | Post-match Press Conference | Wimbledon 2025',
   'content': "What went wrong | Carlos Alcaraz | Post-match Press Conference | Wimbledon 2025\nWimbledon\n2470000 subscribers\n7459 likes\n748819 views\n13 Jul 2025\nSpanish World No.2, Carlos Alcaraz, speaks after his Gentlemen's Singles Final loss to Italy's Jannik Sinner on Centre Court at Wimbledon 2025.\n \n#Wimbledon #Tennis #Interview #Wimbledon2025 #TheresOnlyOneWimbledon \n\nSUBSCRIBE to keep up with all The Championships action and news!  \n\nJoin myWimbledon for a personalised Wimbledon experience: http://wimbledon.com/mywimbledon \n\nTo follow all of the action as it happens go to: https://www.wimbledon.com/en_GB/scores/index.html \n\nTo license Wimbledon footage, visit: https://bit.ly/3fy0RbA\n\n",
   'score': 0.43136916

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-large",  # multilingual support
        input=text
    )
    return np.array(response.data[0].embedding)

for item in faq_data:
    item["embedding"] = get_embedding(item["question"])

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def find_best_match(user_query, threshold=0.70):
    query_vec = get_embedding(user_query)
    best_match = None
    best_score = -1
    
    for item in faq_data:
        score = cosine_similarity(query_vec, item["embedding"])
        if score > best_score:
            best_match = item
            best_score = score
            
    if best_score >= threshold:
        return best_match["answer"], best_score
    else:
        return None, best_score


In [ ]:
def answer_query(user_query):
    preset_answer, score = find_best_match(user_query)
    
    print(preset_answer)
    print(score)

    if preset_answer:
        # Use preset info to guide generation
        prompt = f"The following preset answer seems to match the user's question (similarity {score:.2f}). " \
                 f"Use it as a reference but rewrite naturally if needed.\n\nPreset: {preset_answer}\n\nUser: {user_query}\nAnswer:"
    else:
        # No strong match → generate freely
        prompt = f"User: {user_query}\nAnswer:"
    
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return completion.choices[0].message.content.strip()


In [ ]:
print(answer_query("Is bomb allowed?"))

None
0.6103822521265352
The use of bombs is heavily regulated and prohibited in many contexts, depending on the laws of a country and international treaties. In general, bombs are associated with warfare, terrorism, and illegal activities, and their use is governed by strict legal frameworks. If you're referring to a specific context, such as construction or demolition, there are also regulations in place. It's important to clarify your question or provide more context for a more tailored response.
